# Ch.00 — Data Acquisition and Collection *(Exercise)*

> **Dataset:** SpaceX Falcon 9 launches — REST API v5 + Wikipedia HTML tables
> **Chapter doc:** [data-acquisition.md](data-acquisition.md) | **Solution:** [notebook-solution.ipynb](notebook-solution.ipynb)

Implement every function marked **TODO**. Each `raise NotImplementedError` is one stub.

| # | Function | Concept | Time |
|---|----------|---------|------|
| 1 | `build_session()` | Retry + exponential backoff | 10 min |
| 2 | `fetch_all_launches()` | Offset pagination | 15 min |
| 3 | `LaunchRecord` + `fetch_validated_launches()` | Pydantic schema validation | 20 min |
| 4 | `fetch_with_change_detection()` | Content hashing | 10 min |
| 5 | `scrape_wikipedia_launches()` | BeautifulSoup + post-processing | 20 min |
| 6 | `handle_landing_outcome()` | MNAR treatment | 10 min |
| 7 | `coerce_boolean()` + `coerce_mass_kg()` | Type normalisation | 10 min |
| 8 | `merge_with_priority()` + `validate_launches()` | Source merge + contract | 15 min |
| 9 | `run_pipeline()` | Idempotent assembly | 15 min |


In [ ]:
# %pip install requests pydantic pandas beautifulsoup4 lxml --quiet

import hashlib, json, logging, re
from datetime import datetime
from pathlib import Path
from typing import Optional

import pandas as pd
import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel, field_validator
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

RAW_DIR       = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

LAUNCHES_URL = "https://api.spacexdata.com/v5/launches"
RAW_JSONL    = RAW_DIR / "launches.jsonl"
HASH_STORE   = RAW_DIR / ".content_hashes.json"

print("Setup complete.", RAW_DIR, PROCESSED_DIR)


---
## § 1 — REST API Data Collection

### HTTP status codes

| Code | Meaning | Action |
|------|---------|--------|
| 200 | OK | Parse and persist |
| 400 | Bad request | Fix request; **do not retry** |
| 429 | Rate limited | Back off; respect `Retry-After` |
| 500/503 | Server error | Retry with backoff |

### Exponential backoff

$$t_{wait} = \min\!\left(c \cdot 2^n + \varepsilon,\; t_{max}\right)$$

$n$ = attempt (0-indexed), $c=1$ s, $\varepsilon \sim \text{Uniform}(0,c)$ jitter prevents thundering herd.
`backoff_factor=1.0` → waits 0 s, 2 s, 4 s, 8 s, 16 s.

Reference: data-acquisition.md § 1 — "Rate limiting and exponential backoff"

In [ ]:
def build_session(retries=5, backoff_factor=1.0,
                  status_forcelist=(429, 500, 502, 503, 504)):
    """
    TODO #1: Build a requests.Session with automatic retry and exponential backoff.

    Steps:
    1. Create requests.Session()
    2. Create Retry(total, backoff_factor, status_forcelist,
                    respect_retry_after_header=True)
    3. Create HTTPAdapter(max_retries=retry)
    4. session.mount("https://", adapter); session.mount("http://", adapter)
    5. Return session

    📖 data-acquisition.md § 1 — "Rate limiting and exponential backoff"
    """
    raise NotImplementedError("TODO #1: implement build_session()")


session = build_session()
print("Session:", session)


### Pagination — the silent truncation problem

The SpaceX v5 API defaults to **10** results per call and caps at **100**.
Calling `/v5/launches` with no params returns only the 10 most recent — not the 200+ corpus.

**Offset pattern:** `?offset=0&limit=100` → `?offset=100&limit=100` → ... → empty list = done.

 data-acquisition.md § 1 — "Pagination: the silent truncation problem"

In [ ]:
def fetch_all_launches(url=LAUNCHES_URL, output=RAW_JSONL, limit=100):
    """
    TODO #2: Full-refresh fetch to JSONL using offset pagination.

    Steps:
    1. sess = build_session()
    2. output.parent.mkdir(parents=True, exist_ok=True)
    3. Open output for writing ("w", encoding="utf-8")
    4. Loop:
       a. GET url with params={"offset": offset, "limit": limit}
       b. resp.raise_for_status()
       c. records = resp.json()
       d. if not records: break
       e. write each: f.write(json.dumps(record) + "\\n")
       f. written += len(records); offset += limit
    5. Return written

    📖 data-acquisition.md § 1 — "The complete fetch pattern"
    """
    raise NotImplementedError("TODO #2: implement fetch_all_launches()")


# Preview (does not require your implementation)
sample = requests.get(LAUNCHES_URL, params={"limit": 1}, timeout=10).json()[0]
print("Keys:", list(sample.keys()))


### Schema validation — catching drift at ingest time

SpaceX v3 → v5: `launch_success` renamed to `success`.
A v3-era pipeline silently reads `None` for the target — no exception, just a useless model.

**Fix:** Pydantic validates every record at ingest. `ValidationError` fires immediately.

Key patterns:
- `Optional[bool] = None` — null is legitimate for upcoming launches
- `@field_validator("date_utc", mode="before")` — transform raw value before validation
- `model_dump(mode="json")` — JSON-safe output for JSONL

 data-acquisition.md § 1 — "Schema validation: catching drift at ingest time"

In [ ]:
class LaunchRecord(BaseModel):
    """
    TODO #3a: Define Pydantic model for SpaceX v5 launch records.

    Fields: id:str  name:str  date_utc:datetime  success:Optional[bool]=None
            upcoming:bool  cores:list

    Add @field_validator("date_utc", mode="before"):
        if isinstance(v, str): return datetime.fromisoformat(v.rstrip("Z"))
        return v

    📖 data-acquisition.md § 1 — "Schema validation"
    """
    pass   # TODO: replace with fields + validator


def fetch_validated_launches(url=LAUNCHES_URL, output=RAW_JSONL, limit=100):
    """
    TODO #3b: Like fetch_all_launches but validate each record with LaunchRecord.

    For each raw record:
      try:    launch = LaunchRecord(**raw); write launch.model_dump(mode="json")
      except ValidationError: log.warning(...); skipped += 1
    Return written count.

    📖 data-acquisition.md § 1 — "Schema validation"
    """
    raise NotImplementedError("TODO #3b: implement fetch_validated_launches()")


# Test against a live record
raw = requests.get(LAUNCHES_URL, params={"limit": 1}, timeout=10).json()[0]
try:
    v = LaunchRecord(**raw)
    print(f"id={v.id}  name={v.name}  success={v.success}  upcoming={v.upcoming}")
except Exception as e:
    print(f"{type(e).__name__}: {e}")


In [ ]:
# Fetch all launches into a DataFrame for exploration
records, offset = [], 0
while True:
    batch = requests.get(LAUNCHES_URL,
                         params={"offset": offset, "limit": 100}, timeout=15).json()
    if not batch:
        break
    records.extend(batch)
    offset += 100

df_api = pd.DataFrame([{
    "flight_number": r.get("flight_number"),
    "name":          r.get("name"),
    "date_utc":      r.get("date_utc"),
    "success":       r.get("success"),
    "upcoming":      r.get("upcoming"),
} for r in records])
df_api["date_utc"] = pd.to_datetime(df_api["date_utc"], utc=True)

print(f"Total launches : {len(df_api)}")
print(f"Success rate   : {df_api['success'].mean():.1%}  (excludes null)")
print(f"Null success   : {df_api['success'].isna().sum()}  (upcoming / unknown)")
df_api.head()


---
## § 2 — Web Scraping

SpaceX launches 2010–2014 predate the API. Only Wikipedia has early Falcon 9 test outcomes.

### Parsing hierarchy

```
URL → requests.get() → BeautifulSoup → .find("table", {"class":"wikitable"}) → pd.read_html()
```

**CSS selector fragility:** `soup.find("table", {"class":"wikitable"})` is stable —
`wikitable` is an established MediaWiki convention. Never use positional indexing
(`soup.find_all("table")[3]`) — it breaks on the first table addition.

### Change detection

Hash raw HTML bytes. On first run: store hash. On subsequent runs:
if hash changed → **raise ValueError before parsing** — loud failure, not silent corruption.

 data-acquisition.md § 2 — "Change detection: hash, don't assume"

In [ ]:
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"
def fetch_with_change_detection(url, name):
    """
    TODO #4: Fetch URL; raise ValueError if content hash changed since last run.

    Steps:
    1. requests.get(url, timeout=20, headers={"User-Agent":"research-bot/1.0"})
    2. resp.raise_for_status()
    3. current_hash = hashlib.sha256(resp.content).hexdigest()
    4. hashes = json.loads(HASH_STORE.read_text()) if HASH_STORE.exists() else {}
    5. last_hash = hashes.get(name)
    6. if last_hash and last_hash != current_hash: raise ValueError(...)
    7. hashes[name] = current_hash; HASH_STORE.write_text(json.dumps(hashes, indent=2))
    8. return resp.content

    📖 data-acquisition.md § 2 — "Change detection: hash, don't assume"
    """
    raise NotImplementedError("TODO #4: implement fetch_with_change_detection()")

print("fetch_with_change_detection() stub defined.")


In [ ]:
WIKI_RAW = RAW_DIR / "html" / "wiki_launches.html"
WIKI_RAW.parent.mkdir(parents=True, exist_ok=True)


def scrape_wikipedia_launches(url=WIKI_URL):
    """
    TODO #5: Fetch Wikipedia launch table; return cleaned DataFrame.

    Steps:
    1. html_bytes = fetch_with_change_detection(url, "wiki_falcon9_launches")
    2. WIKI_RAW.write_bytes(html_bytes)
    3. soup = BeautifulSoup(html_bytes, "html.parser")
    4. table = soup.find("table", {"class": "wikitable"})
       if table is None: raise RuntimeError(...)
    5. df = pd.read_html(str(table))[0]
    6. Flatten MultiIndex columns:
       if isinstance(df.columns, pd.MultiIndex):
           df.columns = [" ".join(filter(None, map(str, col))).strip() ...]
    7. df.replace(r"\\[.*?\\]", "", regex=True)   strip citations
    8. df.replace(r"^\\s*$", pd.NA, regex=True)     whitespace -> NA
    9. Keep rows where r.nunique(dropna=False) > 1     drop footnote rows
   10. df.dropna(how="all").reset_index(drop=True)
   11. Return df

    📖 data-acquisition.md § 2 — "Multi-header rows and the post-processing reality"
    """
    raise NotImplementedError("TODO #5: implement scrape_wikipedia_launches()")


# Uncomment after implementing TODO #4 and #5:
# df_wiki = scrape_wikipedia_launches()
# print(df_wiki.shape)
# df_wiki.head(3)
print("scrape_wikipedia_launches() stub defined.")


---
## § 3 — Data Wrangling

### Missing value taxonomy: MCAR / MAR / MNAR

| Mechanism | Definition | SpaceX example | Treatment |
|-----------|-----------|----------------|----------|
| **MCAR** | Missing independent of all variables | 2% API dropout from timeouts | Drop rows (< 5%) or impute with median |
| **MAR** | Missing depends on *other observed* columns | `payload_mass_kg` missing more for early launches (correlates with `year`) | Group imputation |
| **MNAR** | Missing depends on *the value itself* | `landing_outcome` absent when landing **not attempted** | Indicator columns — **do not impute** |

**MNAR critical insight:** imputing `landing_success = False` for not-attempted launches
teaches the model those configurations *failed*. They simply weren't attempted.
Three correct states: attempted+succeeded, attempted+failed, not attempted.

 data-acquisition.md § 3 — "Missing value taxonomy: MCAR, MAR, MNAR"

In [ ]:
def handle_landing_outcome(df):
    """
    TODO #6: MNAR treatment — split 'landing_outcome' into two indicator columns.

    Three states:
    - attempted+succeeded → landing_attempted=True,  landing_success=True
    - attempted+failed    → landing_attempted=True,  landing_success=False
    - not attempted       → landing_attempted=False, landing_success=NA

    Steps:
    1. df = df.copy()
    2. df["landing_attempted"] = df["landing_outcome"].notna()
    3. parse_outcome(v): pd.NA if isna(v); True if lower in success set;
       False if lower in failure set; else pd.NA   (not assumed failure)
    4. df["landing_success"] = df["landing_outcome"].map(parse_outcome).astype(pd.BooleanDtype())
    5. Return df

    📖 data-acquisition.md § 3 — MNAR section
    """
    raise NotImplementedError("TODO #6: implement handle_landing_outcome()")


demo = pd.DataFrame({
    "flight":          [1,        2,        3,    4,        5],
    "landing_outcome": ["Success","Failure", None,"success","Crash"],
})
print(handle_landing_outcome(demo)[
    ["flight", "landing_outcome", "landing_attempted", "landing_success"]
])


In [ ]:
def coerce_boolean(series):
    """
    TODO #7a: Normalise any boolean-ish string to pd.BooleanDtype.

    True  ← 'true','yes','1','success','successful','landed'
    False ← 'false','no','0','failure','failed','crash','n/a','none','nan',''
    pd.NA ← anything else (unrecognised — not assumed)

    Pipeline: .astype(str).str.lower().str.strip().map(lambda...).astype(pd.BooleanDtype())

    📖 data-acquisition.md § 3 — "Type coercion: normalizing the chaos"
    """
    raise NotImplementedError("TODO #7a: implement coerce_boolean()")


def coerce_mass_kg(series):
    """
    TODO #7b: Parse '9525 kg', '9,525' to float.

    1. series.astype(str).str.replace(r'[^\\d.]', '', regex=True)
    2. pd.to_numeric(..., errors='coerce')   ← NaN for unparseable (visible failure)

    📖 data-acquisition.md § 3 — Type coercion table
    """
    raise NotImplementedError("TODO #7b: implement coerce_mass_kg()")


messy = pd.Series(["TRUE", "False", "1", "0", "Success", "N/A", None, "yes", "unknown"])
clean = coerce_boolean(messy)
print(pd.DataFrame({"raw": messy, "coerced": clean}).to_string(index=False))
print(f"\nNull (unrecognised): {clean.isna().sum()}")


### Deduplication and source-priority merging

The same launch can appear in API and Wikipedia with **different values** — one source was
retroactively corrected. Define priority per field, document it, use `combine_first()`.

`combine_first(other)` — takes left value when non-null, falls back to right.

### Validation contract

Write assertions **before** writing wrangling code.
If assertions fail, pipeline halts with a clear error — not a silent corrupt output.

 data-acquisition.md § 3 — "Deduplication" + "Validation before downstream use"

In [ ]:
def merge_with_priority(api_df, csv_df, key="flight_number"):
    """
    TODO #8a: Outer join with explicit source priority per field.

    Steps:
    1. merged = api_df.merge(csv_df, on=key, suffixes=("_api","_csv"), how="outer")
    2. merged["success"] = merged["success_api"].combine_first(merged["success_csv"])
       drop "success_api", "success_csv"
    3. merged["launch_site"] = merged["launch_site_csv"].combine_first(merged["launch_site_api"])
       drop suffixed columns
    4. Return merged

    📖 data-acquisition.md § 3 — "Deduplication: exact vs near-duplicate"
    """
    raise NotImplementedError("TODO #8a: implement merge_with_priority()")


def validate_launches(df):
    """
    TODO #8b: Assert data quality invariants before writing to data/processed/.

    Assert:
    1. len(df) >= 50
    2. 'flight_number' and 'name' have zero nulls
    3. 'flight_number' has zero duplicates

    Each assert must carry a descriptive message with the actual value.

    📖 data-acquisition.md § 3 — "Validation before downstream use"
    """
    raise NotImplementedError("TODO #8b: implement validate_launches()")


try:
    validate_launches(df_api)
    print("Validation passed ✅")
except (AssertionError, NotImplementedError) as e:
    print(f"{type(e).__name__}: {e}")


---
## § 4 — Reproducible Pipeline

### Three enemies of reproducibility
1. **Mutable sources** — APIs return different data on different days
2. **Local state** — files the pipeline reads that aren't in version control
3. **Implicit ordering** — step 3 silently reads stale intermediate data

### Layout rule
```
data/
├── raw/ ← append-only; never overwritten
└── processed/ ← derivable from raw; disposable
```

**Idempotency:** running twice = identical output. Hash-gated fetches make this cheap.

 data-acquisition.md § 4 — "Idempotency" + "Raw vs processed"

In [ ]:
def run_pipeline():
    """
    TODO #9: Idempotent end-to-end pipeline.

    Stage 1  fetch:    fetched = fetch_validated_launches(output=RAW_JSONL)
    Stage 2  load:     df = pd.read_json(RAW_JSONL, lines=True)
                       log shape and null counts
    Stage 3  wrangle:  date_utc → datetime; success → coerce_boolean; dedup
    Stage 4  validate: validate_launches(df); raise on failure
    Stage 5  persist:  df.to_parquet(PROCESSED_DIR / "launches_clean.parquet")

    Log row counts at each stage. Return final DataFrame.

    📖 data-acquisition.md § 4 — "Logging and observability"
    """
    log.info("=== Pipeline start ===")
    raise NotImplementedError("TODO #9: implement run_pipeline()")


# Uncomment after implementing all TODOs:
# df_final = run_pipeline()
# df_final.head()
print("run_pipeline() stub defined.")


In [ ]:
# Fetch all and run wrangling inline (works even before TODOs are done)
records, offset = [], 0
while True:
    batch = requests.get(LAUNCHES_URL,
                         params={"offset": offset, "limit": 100}, timeout=15).json()
    if not batch:
        break
    records.extend(batch)
    offset += 100

df_check = pd.DataFrame([{
    "flight_number": r.get("flight_number"),
    "name":          r.get("name"),
    "date_utc":      r.get("date_utc"),
    "success":       r.get("success"),
} for r in records])
df_check["date_utc"] = pd.to_datetime(df_check["date_utc"], utc=True)

print(f"Records   : {len(df_check)}")
print(f"Date range: {df_check['date_utc'].min().date()} → {df_check['date_utc'].max().date()}")
print(f"Null success: {df_check['success'].isna().sum()}")
print()
# Uncomment after implementing coerce_boolean():
# df_check['success_bool'] = coerce_boolean(df_check['success'].astype(str))
# print('Success rate:', df_check['success_bool'].mean())
df_check.head()


---
## Checklist

- [ ] `build_session()` — no `NotImplementedError`; session printed
- [ ] `fetch_all_launches()` — row count > 10 (more than page one)
- [ ] `LaunchRecord` — validates a live API record without error
- [ ] `fetch_with_change_detection()` — stores hash on first run; raises on content change
- [ ] `scrape_wikipedia_launches()` — DataFrame with flattened columns, no `[1]` markers
- [ ] `handle_landing_outcome()` — demo shows 3-column MNAR result correctly
- [ ] `coerce_boolean()` — `"unknown"` → `pd.NA`, not `True`/`False`
- [ ] `validate_launches()` — passes on `df_api` from the summary cell
- [ ] `run_pipeline()` — writes `data/processed/launches_clean.parquet`

## Exercise connection

| What you built | Used downstream |
|---|---|
| `data/processed/launches_clean.parquet` | Input to `exercises/01-ml/01-regression/src/data-prep.py` |
| `validate_launches()` pattern | Template for `data-prep.py` TODOs #11–12 |
| `coerce_boolean()` | Extended in `data-prep.py` for SmartVal AI |

**Solution:** [notebook-solution.ipynb](notebook-solution.ipynb)